This notebook creates all plots regarding the slopiness analysis, allowing to estimate the influence of each parameter in the model. For the mutant search algorithm, it ranks the binding to which IL-2 receptor is important to separate a group of target and non-target cells.

In [1]:
from sympy import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import use
use('Agg')
import warnings
warnings.filterwarnings('ignore')
plt.rc('xtick', labelsize=14)
plt.rc('ytick', labelsize=14)

# Plot results of slopiness with raw parameter data

In [4]:
# Load Hessian
H = np.loadtxt('results/Hessian_nofit.csv', delimiter=',')
param_vec = ['$V_{EC}$','$S_{cell}$','$V_{cell}$','$Kd_{IL-RA}$','$Kd_{IL-RB}$','$Kd_{IL-RG}$','$Kd_{IL-RA-RB}$','$Kd_{IL-RB-RG}$','$Kd_{IL-RA-RB-RG}$','$Kd_{STAT5}$','$K_{PHOS}$','$K_{DEPHOS}$']
df = pd.DataFrame(H, columns = param_vec, index = param_vec)

In [5]:
# First row and column are removed since vol_EC has no effect on the model (as it should!)
param_vec = ['vol_EC','surf_cell','vol_cell','k_IL_RA_b','k_IL_RB_b','k_IL_RG_b','k_IL_RA_RB_b','k_IL_RB_RG_b','k_IL_RA_RB_RG_b','k_STAT5_b','k_PHOS','k_DEPHOS']

# Perform the eigendecomposition of the Hessian
eigenvalues, eigenvectors = np.linalg.eig(H)

# Sort eigenvalues from big to small
eigenvalues_sorted = np.sort(eigenvalues,)[::-1]
eigenvectors_sorted = []
for eigv in eigenvalues_sorted:
    i = 0
    for eigv2 in eigenvalues:
        if eigv == eigv2:
            break
        i += 1
    eigenvectors_sorted.append(eigenvectors[:, i]/np.linalg.norm(eigenvectors[i]))

# Using symbolic math get the strongest eigenparameters
param_symbols = []
for param in param_vec:
    param_symbols.append(symbols(param))
    
# Get formula of the eigenvectors
eigenparameters = []
for vector in eigenvectors_sorted:
    eigenparam_components = []
    i = 0
    for param in param_symbols:
        if abs(vector[i]) >= 0.1:
            eigenparam_components.append(param_symbols[i]**round(vector[i], 3))
        i += 1
    eigenparam = eigenparam_components[0]
    i = 1
    for eigenparam_component in eigenparam_components[1:]:
        eigenparam = eigenparam*eigenparam_components[i]
        i += 1
    eigenparameters.append(eigenparam)
print("Normalized eigenparameters:")
i = 0
for eigenparam in eigenparameters:
    print("Eigenvalue: "+ str(eigenvalues_sorted[i]))
    print("Eigenparameter: " + str(eigenparameters[i]))
    i += 1

Normalized eigenparameters:
Eigenvalue: 0.0006347521446784467
Eigenparameter: k_IL_RA_b**0.187*k_IL_RB_b**0.925*k_IL_RG_b**0.103*surf_cell**0.295
Eigenvalue: 0.00012016184852141496
Eigenparameter: k_IL_RA_RB_b**0.351*k_IL_RA_b**0.79*k_PHOS**0.108*surf_cell**0.356/k_IL_RB_b**0.307
Eigenvalue: 3.1774583708848954e-05
Eigenparameter: k_IL_RA_b**0.225*k_IL_RG_b**0.942/surf_cell**0.163
Eigenvalue: 3.7941415991749712e-06
Eigenparameter: k_IL_RG_b**0.206*surf_cell**0.845/(k_IL_RA_b**0.415*k_IL_RB_b**0.203*k_PHOS**0.125)
Eigenvalue: 2.643711384256617e-07
Eigenparameter: k_DEPHOS**0.558*k_IL_RA_RB_b**0.183*k_IL_RA_b**0.132/(k_IL_RG_b**0.16*k_PHOS**0.706*k_STAT5_b**0.126*vol_cell**0.274)
Eigenvalue: 2.509006113276835e-08
Eigenparameter: k_IL_RA_b**0.316*surf_cell**0.195/(k_IL_RA_RB_b**0.907*k_IL_RG_b**0.154)
Eigenvalue: 5.740948824278731e-10
Eigenparameter: k_DEPHOS**0.193*k_IL_RA_RB_RG_b**0.676*k_IL_RB_RG_b**0.676*vol_cell**0.18
Eigenvalue: 1.2622271318353761e-10
Eigenparameter: k_DEPHOS**0.42*k

In [6]:
param_vec = ['$V_{EC}$','$S_{cell}$','$V_{cell}$','$Kd_{IL-RA}$','$Kd_{IL-RB}$','$Kd_{IL-RG}$','$Kd_{IL-RA-RB}$','$Kd_{IL-RB-RG}$','$Kd_{IL-RA-RB-RG}$','$Kd_{STAT5}$','$K_{PHOS}$','$K_{DEPHOS}$']
# Plot the second derivative of each parameter to see which has more influence on the cost by itself
Hii = [df.loc[param,param] for param in param_vec]
fig,ax=plt.subplots(1,1,figsize=(8, 8), dpi=400)
param_vec = ['$V_{EC}$','$\mathbf{S_{cell}}$','$V_{cell}$','$\mathbf{Kd_{IL-RA}}$','$\mathbf{Kd_{IL-RB}}$','$\mathbf{Kd_{IL-RG}}$','$\mathbf{Kd_{IL-RA-RB}}$','$Kd_{IL-RB-RG}$','$Kd_{IL-RA-RB-RG}$','$Kd_{STAT5}$','$K_{PHOS}$','$K_{DEPHOS}$']
plt.bar(param_vec, eigenvectors_sorted[0]*eigenvalues_sorted[0], width=0.9, color="darkred", alpha=0.8, label="Eigenvalue 1")
plt.bar(param_vec, eigenvectors_sorted[1]*eigenvalues_sorted[1], width=0.9, color="darkblue", alpha=0.8, label="Eigenvalue 2")
plt.bar(param_vec, eigenvectors_sorted[2]*eigenvalues_sorted[2], width=0.9, color="darkgreen", alpha=0.8, label="Eigenvalue 3")
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_tick_params(width=5, length=10)
ax.xaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=22)
ax.tick_params(axis='y', labelsize=22)
plt.ylabel('$\lambda \cdot v$', fontsize=25)
plt.xticks(rotation=45, ha="right")
ax.legend(loc="upper right", fontsize=15)
plt.savefig('figures/slopiness/slopiness_raw_param_eigenval.pdf', bbox_inches='tight', transparent=True)
plt.show()

# Plot results of slopiness for a novel mutant in melanoma treatment

In [2]:
# Load Hessian
H = np.loadtxt('results/Hessian_target_melanoma_IL2s.csv', delimiter=',')
param_vec = ['$Kd_{IL-RA}$','$Kd_{IL-RB}$','$Kd_{IL-RG}$','$Kd_{IL-RA-RB}$','$Kd_{IL-RB-RG}$','$Kd_{IL-RA-RB-RG}$']
df = pd.DataFrame(H, columns = param_vec, index = param_vec)

In [3]:
# First row and column are removed since vol_EC has no effect on the model (as it should!)
param_vec = ['k_IL_RA_b','k_IL_RB_b','k_IL_RG_b','k_IL_RA_RB_b','k_IL_RB_RG_b','k_IL_RA_RB_RG_b']

# Perform the eigendecomposition of the Hessian
eigenvalues, eigenvectors = np.linalg.eig(H)

# Sort eigenvalues from big to small
eigenvalues_sorted = np.sort(eigenvalues,)[::-1]
eigenvectors_sorted = []
for eigv in eigenvalues_sorted:
    i = 0
    for eigv2 in eigenvalues:
        if eigv == eigv2:
            break
        i += 1
    eigenvectors_sorted.append(eigenvectors[:, i]/np.linalg.norm(eigenvectors[i]))

# Using symbolic math get the strongest eigenparameters
param_symbols = []
for param in param_vec:
    param_symbols.append(symbols(param))
    
# Get formula of the eigenvectors
eigenparameters = []
for vector in eigenvectors_sorted:
    eigenparam_components = []
    i = 0
    for param in param_symbols:
        if abs(vector[i]) >= 0.1:
            eigenparam_components.append(param_symbols[i]**round(vector[i], 3))
        i += 1
    eigenparam = eigenparam_components[0]
    i = 1
    for eigenparam_component in eigenparam_components[1:]:
        eigenparam = eigenparam*eigenparam_components[i]
        i += 1
    eigenparameters.append(eigenparam)
print("Normalized eigenparameters:")
i = 0
for eigenparam in eigenparameters:
    print("Eigenvalue: "+ str(eigenvalues_sorted[i]))
    print("Eigenparameter: " + str(eigenparameters[i]))
    i += 1

Normalized eigenparameters:
Eigenvalue: 6123428.740630075
Eigenparameter: 1/(k_IL_RB_b**0.713*k_IL_RG_b**0.7)
Eigenvalue: 167.06723225390553
Eigenparameter: 1/(k_IL_RA_b**0.987*k_IL_RG_b**0.136)
Eigenvalue: 0.18985338292933143
Eigenparameter: k_IL_RA_b**0.155*k_IL_RB_b**0.695/k_IL_RG_b**0.7
Eigenvalue: 2.118665174954911e-05
Eigenparameter: k_IL_RA_RB_b**0.996
Eigenvalue: 2.629942470497519e-06
Eigenparameter: 1/(k_IL_RA_RB_RG_b**0.706*k_IL_RB_RG_b**0.706)
Eigenvalue: -5.764767259665655e-40
Eigenparameter: k_IL_RA_RB_RG_b**0.707/k_IL_RB_RG_b**0.707


In [4]:
param_vec = ['$Kd_{IL-RA}$','$Kd_{IL-RB}$','$Kd_{IL-RG}$','$Kd_{IL-RA-RB}$','$Kd_{IL-RB-RG}$','$Kd_{IL-RA-RB-RG}$']# Plot the second derivative of each parameter to see which has more influence on the cost by itself
Hii = [df.loc[param,param] for param in param_vec]
fig,ax=plt.subplots(1,1,figsize=(8, 8), dpi=400)
param_vec = ['$\mathbf{Kd_{IL-RA}}$','$\mathbf{Kd_{IL-RB}}$','$\mathbf{Kd_{IL-RG}}$','$Kd_{IL-RA-RB}$','$Kd_{IL-RB-RG}$','$Kd_{IL-RA-RB-RG}$']
plt.bar(param_vec, eigenvectors_sorted[0]*eigenvalues_sorted[0], width=0.9, color="darkred", alpha=0.8, label="Eigenvalue 1")
plt.bar(param_vec, eigenvectors_sorted[1]*eigenvalues_sorted[1], width=0.9, color="darkblue", alpha=0.8, label="Eigenvalue 2")
plt.bar(param_vec, eigenvectors_sorted[2]*eigenvalues_sorted[2], width=0.9, color="darkgreen", alpha=0.8, label="Eigenvalue 3")
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_tick_params(width=5, length=10)
ax.xaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=22)
ax.tick_params(axis='y', labelsize=22)
plt.ylabel('$\lambda \cdot v$', fontsize=25)
plt.xticks(rotation=45, ha="right")
ax.legend(loc="upper right", fontsize=15)
plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
plt.savefig('figures/slopiness/slopiness_melanoma_eigenval.pdf', bbox_inches='tight', transparent=True)
plt.show()